In [1]:
# Setup — imports, paths, test data
import os
import pandas as pd
import numpy as np
import pydicom
import cv2
import torch
import torch.nn as nn
from transformers import Dinov2Model

data_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
dinov2_path = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"

test = pd.read_csv(f"{data_dir}/test.csv")
test_series = pd.read_csv(f"{data_dir}/test_series.csv")

finding_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 
                    'Contusion', 'Fracture']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Test studies: {len(test)}")
print(f"Device: {device}")

Test studies: 3
Device: cpu


In [2]:
# Helper functions — 5-slot series selection, mm-crop, RGB-grouping, physical-slice-ordering
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1)]

def select_series_5slots(study_id, series_df):
    study_series = series_df[series_df['StudyInstanceUID'] == study_id]
    slots = {}
    for plane, fluid_sensitive in SLOTS:
        key = f"{plane}_{fluid_sensitive}"
        candidates = study_series[
            (study_series['Anatomical_Plane'] == plane) & 
            (study_series['Fluid_Sensitive'] == fluid_sensitive)
        ]
        slots[key] = candidates.iloc[0]['SeriesInstanceUID'] if len(candidates) > 0 else None
    return slots

def crop_by_mm(pixel_array, pixel_spacing, crop_mm=130.0):
    row_spacing, col_spacing = pixel_spacing
    crop_h = int(crop_mm / row_spacing)
    crop_w = int(crop_mm / col_spacing)
    h, w = pixel_array.shape
    center_h, center_w = h // 2, w // 2
    half_h, half_w = crop_h // 2, crop_w // 2
    top = max(0, center_h - half_h)
    bottom = min(h, center_h + half_h)
    left = max(0, center_w - half_w)
    right = min(w, center_w + half_w)
    return pixel_array[top:bottom, left:right]

def crop_and_resize(pixel_array, pixel_spacing, crop_mm=130.0, final_size=224):
    cropped = crop_by_mm(pixel_array, pixel_spacing, crop_mm)
    return cv2.resize(cropped, (final_size, final_size), interpolation=cv2.INTER_LINEAR)

def select_slice_indices_grouped(total_slices, n_groups=16, band=(0.12, 0.88)):
    start = int(total_slices * band[0])
    end = int(total_slices * band[1])
    center_indices = np.linspace(start, end - 1, n_groups).astype(int)
    groups = []
    for center in center_indices:
        prev_idx = max(0, center - 1)
        next_idx = min(total_slices - 1, center + 1)
        groups.append((prev_idx, center, next_idx))
    return groups

def load_rgb_group(slices_sorted, group_indices, pixel_spacing, crop_mm=130.0, final_size=224):
    channels = []
    for idx in group_indices:
        arr = slices_sorted[idx].pixel_array.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
        cr = crop_and_resize(arr, pixel_spacing, crop_mm, final_size)
        channels.append(cr)
    return np.stack(channels, axis=-1)

def get_position_key(d):
    try:
        orientation = [float(x) for x in d.ImageOrientationPatient]
        row_vec = np.array(orientation[:3])
        col_vec = np.array(orientation[3:])
        normal = np.cross(row_vec, col_vec)
        position = np.array([float(x) for x in d.ImagePositionPatient])
        return np.dot(position, normal)
    except Exception:
        return int(d.InstanceNumber)

print("Helper functions ready")

Helper functions ready


In [3]:
# Model classes — SliceAttentionPool + DINOv2KneeModelV2 (3-channel input)
class SliceAttentionPool(nn.Module):
    def __init__(self, dim=384):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return (x * weights).sum(dim=1)

class DINOv2KneeModelV2(nn.Module):
    def __init__(self, dinov2_path, num_findings=12, n_slots=5, n_groups=16):
        super().__init__()
        self.dinov2 = Dinov2Model.from_pretrained(dinov2_path)
        self.dim = self.dinov2.config.hidden_size
        self.n_slots = n_slots
        self.n_groups = n_groups
        self.slice_pool = SliceAttentionPool(self.dim)
        self.slot_attn = nn.Sequential(nn.Linear(self.dim, 128), nn.Tanh(), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Linear(self.dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_findings))
    
    def forward(self, slots, slot_mask):
        batch_size = slots.shape[0]
        x = slots.view(batch_size * self.n_slots * self.n_groups, 3, 224, 224)
        features = self.dinov2(pixel_values=x).pooler_output
        features = features.view(batch_size, self.n_slots, self.n_groups, self.dim)
        
        slot_features = torch.stack([self.slice_pool(features[:, s, :, :]) for s in range(self.n_slots)], dim=1)
        
        slot_weights = self.slot_attn(slot_features).squeeze(-1)
        slot_weights = slot_weights.masked_fill(slot_mask == 0, float('-inf'))
        slot_weights = torch.softmax(slot_weights, dim=1).unsqueeze(-1)
        
        final_feature = (slot_features * slot_weights).sum(dim=1)
        return self.head(final_feature)

print("Model class ready")

Model class ready


In [4]:

checkpoint_dir = "/kaggle/input/datasets/mayankptdr/dinov2-v2-checkpoints"
print(os.listdir(checkpoint_dir))

['fold2_best', 'fold1_best', 'fold3_best', 'fold4_best', 'fold0_best']


In [5]:
# Cell: Check actual file extensions
for f in os.listdir(checkpoint_dir):
    full_path = f"{checkpoint_dir}/{f}"
    print(f, os.path.getsize(full_path), "bytes")

fold2_best 0 bytes
fold1_best 0 bytes
fold3_best 0 bytes
fold4_best 0 bytes
fold0_best 0 bytes


In [6]:
# Rebuild valid .pt files from Kaggle's auto-extracted folder structure
import zipfile

restored_dir = "/kaggle/working/restored_checkpoints"
os.makedirs(restored_dir, exist_ok=True)

base = "/kaggle/input/datasets/mayankptdr/dinov2-v2-checkpoints"

for fold_num in range(5):
    name = f"fold{fold_num}_best"
    inner_dir = f"{base}/{name}/{name}"  # doubly-nested folder jisme .data aur data hain
    output_pt = f"{restored_dir}/{name}.pt"
    
    with zipfile.ZipFile(output_pt, 'w', zipfile.ZIP_STORED) as zf:
        for root, dirs, files in os.walk(inner_dir):
            for file in files:
                file_path = os.path.join(root, file)
                rel_path = os.path.relpath(file_path, f"{base}/{name}")
                zf.write(file_path, rel_path)
    
    size_mb = os.path.getsize(output_pt) / (1024*1024)
    print(f"Rebuilt {output_pt} ({size_mb:.1f} MB)")

Rebuilt /kaggle/working/restored_checkpoints/fold0_best.pt (85.0 MB)
Rebuilt /kaggle/working/restored_checkpoints/fold1_best.pt (85.0 MB)
Rebuilt /kaggle/working/restored_checkpoints/fold2_best.pt (85.0 MB)
Rebuilt /kaggle/working/restored_checkpoints/fold3_best.pt (85.0 MB)
Rebuilt /kaggle/working/restored_checkpoints/fold4_best.pt (85.0 MB)


In [7]:
# Load models from rebuilt checkpoints
models = []
for fold_num in range(5):
    m = DINOv2KneeModelV2(dinov2_path, num_findings=12).to(device)
    ckpt = torch.load(f"{restored_dir}/fold{fold_num}_best.pt", map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    models.append(m)
    print(f"Loaded fold{fold_num}, AUROC={ckpt['auroc']:.4f}")

print(f"\nTotal models loaded: {len(models)}")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold0, AUROC=0.7753


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold1, AUROC=0.8051


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold2, AUROC=0.7670


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold3, AUROC=0.7764


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold4, AUROC=0.7818

Total models loaded: 5


In [8]:
# Load slots for one test study — v2 format (RGB-grouped, physical-ordering)
def load_study_slots_test_v2(study_id):
    slot_map = select_series_5slots(study_id, test_series)
    
    slot_volumes = []
    slot_masks = []
    
    for plane, fs in SLOTS:
        key = f"{plane}_{fs}"
        series_id = slot_map[key]
        
        if series_id is None:
            vol = np.zeros((16, 224, 224, 3), dtype=np.float32)
            mask = 0.0
        else:
            folder = f"{data_dir}/test_series/{study_id}/{series_id}"
            dcm_files = os.listdir(folder)
            slices = [pydicom.dcmread(f"{folder}/{f}") for f in dcm_files]
            slices_sorted = sorted(slices, key=get_position_key)
            total_slices = len(slices_sorted)
            pixel_spacing = [float(x) for x in slices_sorted[0].PixelSpacing]
            groups = select_slice_indices_grouped(total_slices, 16)
            
            rgb_groups = []
            for group_indices in groups:
                rgb = load_rgb_group(slices_sorted, group_indices, pixel_spacing, 130.0, 224)
                rgb_groups.append(rgb)
            vol = np.stack(rgb_groups).astype(np.float32)
            mask = 1.0
        
        slot_volumes.append(vol)
        slot_masks.append(mask)
    
    slots_array = np.stack(slot_volumes)  # (5, 16, 224, 224, 3)
    mask_array = np.array(slot_masks, dtype=np.float32)
    
    return slots_array, mask_array

print("Function ready")

Function ready


In [9]:
# Ensemble prediction for one study
def predict_study_ensemble_v2(study_id):
    slots_array, mask_array = load_study_slots_test_v2(study_id)
    
    # (5, 16, 224, 224, 3) -> (5, 16, 3, 224, 224) for PyTorch
    slots_array = np.transpose(slots_array, (0, 1, 4, 2, 3))
    
    slots = torch.tensor(slots_array, dtype=torch.float32).unsqueeze(0).to(device)
    mask = torch.tensor(mask_array, dtype=torch.float32).unsqueeze(0).to(device)
    
    all_probs = []
    with torch.no_grad():
        for m in models:
            out = torch.sigmoid(m(slots, mask))
            all_probs.append(out.float().cpu().numpy())
    
    avg_probs = np.mean(all_probs, axis=0)[0]
    return avg_probs

print("Ensemble function ready")

Ensemble function ready


In [10]:
# Run predictions on test studies, build submission
predictions = []
for study_id in test['StudyInstanceUID']:
    try:
        probs = predict_study_ensemble_v2(study_id)
        predictions.append([study_id] + list(probs))
        print(f"Predicted: {study_id}")
    except Exception as e:
        print(f"Error on {study_id}: {e}")
        predictions.append([study_id] + [0.5]*12)

submission = pd.DataFrame(predictions, columns=['StudyInstanceUID'] + finding_columns)
print(submission)

Predicted: 1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191
Predicted: 1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776
Predicted: 1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.213100  0.222164   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.518460  0.501893   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.292794  0.378722   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.342370          0.258631   0.421172    0.367721  0.505002   
1         0.515972          0.485207   0.542425    0.487800  0.568395   
2         0.497694          0.409038   0.589083    0.517246  0.625457   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.313945   0.276237  0.387764   0.250479  0.048641  
1  0.553098   0.511343  0.499268   0.558615  0.370926  
2  0.4

In [11]:
# Sample verfication with submission formate
sample_sub = pd.read_csv(f"{data_dir}/sample_submission.csv")
print(sample_sub.columns.tolist() == submission.columns.tolist())
print(sample_sub.shape, submission.shape)

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Saved!")

True
(3, 13) (3, 13)
Saved!
